# M-Lab India — Gaussian Mixture Model network-quality profiles

A second, independent take on the same question as `clustering.ipynb` (k-means): group the cleaned
measurements (from `outlier.ipynb`) into a small number of representative "network condition"
profiles, this time with a **Gaussian Mixture Model (GMM)**. Unlike k-means, GMM is probabilistic --
each row gets a probability of belonging to each profile, not just a hard label -- which lets us
also report how much the profiles overlap. Run separately for download and upload; all the machinery
is defined once and reused for both.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import RobustScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'mlab_analysis':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get('MLAB_OUTPUT_DIR', PROJECT_ROOT / 'outputs' / 'analysis' / '2026-06-18_to_2026-11-04'))
FIGURE_DIR = DATA_DIR / 'figures' / 'gmm'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE, GREY, RED, GREEN = '#4C78A8', '#F58518', '#B0B0B0', '#E45756', '#54A24B'
# Okabe-Ito colour-blind-safe palette, ordered best (bluish green) -> worst (reddish purple),
# used everywhere a cluster/profile needs its own colour (scatter plots, profile bar charts).
CLUSTER_PALETTE = ['#009E73', '#56B4E9', '#F0E442', '#E69F00', '#D55E00', '#CC79A7']
RANDOM_STATE = 42

download = pd.read_csv(DATA_DIR / 'download_clean.csv', low_memory=False)
upload = pd.read_csv(DATA_DIR / 'upload_clean.csv', low_memory=False)
DOWNLOAD_FEATURES = ['download_mbps', 'min_rtt_ms', 'loss_rate']
UPLOAD_FEATURES = ['upload_mbps', 'min_rtt_ms']

print(f'Clean download: {download.shape}   Clean upload: {upload.shape}')

## Method (for the methodology slide)

1. **Row-level cleaning for the GMM features.** `outlier.ipynb` already removed data-centre/enterprise
   traffic and invalid rows from the whole dataset; here we additionally drop any row that's missing
   or non-finite on *just the features this model uses*, since GMM (unlike tree-based methods) can't
   fit on `NaN`. Genuine poor-network rows -- low throughput, high RTT/loss -- are finite values, so
   they're kept; only rows that are actually missing a feature are dropped.
2. **Transform** each feature with `log1p` (throughput and RTT are heavily right-skewed; `log1p`
   compresses the tail and is safe at zero).
3. **Scale** with `RobustScaler` (centers on the median, scales by the interquartile range) instead of
   `StandardScaler` -- it's less influenced by the extreme tail values this dataset is known to have
   (see `outlier.ipynb`'s statistical-tail review), which matters more for GMM since covariance
   estimates are more tail-sensitive than k-means' distance-to-centroid.
4. **Fit a GMM for k = 2..6** (`covariance_type='full'`, fixed `random_state=42`, `n_init=5`).
5. **Score each k** three ways: **BIC** and **AIC** (both penalize model complexity; lower is better --
   BIC penalizes it more heavily and is generally preferred for choosing the number of components) and
   **silhouette score** (computed on hard labels from `predict`, on a fixed 5,000-row subsample for the
   same computational reason as in `clustering.ipynb`).
6. **Pick k** mainly from BIC (smallest k within 1% of the BIC range from the minimum -- avoids picking
   a needlessly complex mixture when the improvement is marginal), cross-checked against AIC and
   interpretability, not by silhouette alone.
7. **Rank profiles** by a composite z-scored score on the cluster medians (higher throughput is better,
   lower RTT/loss is worse) -- same approach as the k-means notebook, so the two are comparable.
8. **Report overlap** via each row's posterior probabilities (`predict_proba`): the average
   max-probability (confidence) per profile, and the share of rows below a 0.70 confidence threshold --
   these are the measurements sitting in the "grey zone" between two profiles, which a hard k-means
   label would hide.

In [ ]:
def prepare_features(df, features):
    '''Drop rows missing/non-finite on the clustering features themselves -- genuine extreme (but
    finite) poor-network rows are kept, only actually-missing values are dropped.'''
    clean = df.dropna(subset=features).copy()
    clean = clean[np.isfinite(clean[features]).all(axis=1)]
    dropped = len(df) - len(clean)
    print(f'{dropped} row(s) dropped for missing/non-finite {features} '
          f'(kept, unchanged: all genuine extreme poor-network measurements).')
    return clean

def transform_and_scale(df, features):
    '''log1p each feature (fixes right skew, safe at 0), then RobustScaler (median/IQR --
    less sensitive to the extreme tail than StandardScaler's mean/std).'''
    log_values = np.log1p(df[features])
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(log_values)
    return X_scaled, scaler

def evaluate_gmm_k_range(X_scaled, k_range=range(2, 7), silhouette_sample_size=5000):
    rows, models = [], {}
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(len(X_scaled), size=min(silhouette_sample_size, len(X_scaled)), replace=False)
    for k in k_range:
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=RANDOM_STATE, n_init=5)
        gmm.fit(X_scaled)
        labels = gmm.predict(X_scaled)
        sil = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
        rows.append({'k': k, 'bic': gmm.bic(X_scaled), 'aic': gmm.aic(X_scaled), 'silhouette': sil})
        models[k] = gmm
    return pd.DataFrame(rows), models

def plot_k_selection(k_eval, dataset_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(k_eval['k'], k_eval['bic'], marker='o', color=BLUE, label='BIC')
    axes[0].plot(k_eval['k'], k_eval['aic'], marker='o', color=ORANGE, label='AIC')
    axes[0].set(title=f'{dataset_name.title()}: BIC / AIC vs. k', xlabel='k', ylabel='Information criterion (lower is better)')
    axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(k_eval['k'], k_eval['silhouette'], marker='o', color=GREEN)
    axes[1].set(title=f'{dataset_name.title()}: silhouette score vs. k', xlabel='k', ylabel='Silhouette score')
    axes[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f'{dataset_name}_k_selection.png', dpi=180); plt.show()

def choose_k(k_eval, tolerance_frac=0.01):
    '''Smallest k whose BIC is within `tolerance_frac` of the BIC range from the minimum -- favors a
    simpler, more interpretable mixture when the statistical improvement from adding components is
    marginal, instead of always taking the numerical minimum.'''
    bic_range = k_eval['bic'].max() - k_eval['bic'].min()
    tol = tolerance_frac * bic_range if bic_range > 0 else 0
    best_bic_row = k_eval.loc[k_eval['bic'].idxmin()]
    best_aic_k = int(k_eval.loc[k_eval['aic'].idxmin(), 'k'])
    candidates = k_eval[k_eval['bic'] <= best_bic_row['bic'] + tol]
    chosen = int(candidates['k'].min())
    print(f"Best BIC: k={int(best_bic_row['k'])} (BIC={best_bic_row['bic']:.1f}); best AIC: k={best_aic_k}.")
    print(f"Selected k = {chosen} (smallest k within {100*tolerance_frac:.0f}% of the BIC range of the minimum).")
    return chosen

def build_gmm_cluster_report(df, features, gmm, X_scaled, scaler=None):
    '''If `scaler` is given, also back-transform each component's mean (gmm.means_, in scaled
    log1p space) into original units as `centroid_<feature>` -- the GMM analogue of a k-means
    centroid, used to mark each component's centre on the pairwise scatter plots below.'''
    labels = gmm.predict(X_scaled)
    working = df[features + ['measurement_id', 'client_asn', 'client_as_name', 'state']].copy()
    working['cluster'] = labels
    n = len(working)
    rows = []
    for c in range(gmm.n_components):
        sub = working[working['cluster'] == c]
        row = {'cluster': c, 'count': len(sub), 'percent': round(100 * len(sub) / n, 2)}
        if scaler is not None:
            centroid_log = scaler.inverse_transform(gmm.means_[c].reshape(1, -1))[0]
            centroid_orig = np.expm1(centroid_log)
        for i, feature in enumerate(features):
            row[f'mean_{feature}'] = sub[feature].mean()
            row[f'median_{feature}'] = sub[feature].median()
            if scaler is not None:
                row[f'centroid_{feature}'] = centroid_orig[i]
        rows.append(row)
    return pd.DataFrame(rows), working

def rank_profiles(report, throughput_col, rtt_col='min_rtt_ms', loss_col=None,
                   profile_names=('Excellent', 'Good', 'Fair', 'Poor', 'Very Poor', 'Critical')):
    '''Composite quality score from z-scored MEDIANS across clusters -- same approach as
    clustering.ipynb, so the k-means and GMM profiles are directly comparable.'''
    report = report.copy()
    def z(series):
        return (series - series.mean()) / series.std(ddof=0) if series.std(ddof=0) > 0 else series * 0
    score = z(report[f'median_{throughput_col}']) - z(report[f'median_{rtt_col}'])
    if loss_col:
        score = score - z(report[f'median_{loss_col}'])
    report['quality_score'] = score
    report = report.sort_values('quality_score', ascending=False).reset_index(drop=True)
    report['profile'] = profile_names[:len(report)]
    single_axis_order = report.sort_values(f'median_{throughput_col}', ascending=False)['cluster'].tolist()
    composite_order = report.sort_values('quality_score', ascending=False)['cluster'].tolist()
    if single_axis_order != composite_order:
        print('Note: ranking by throughput alone would have ordered clusters differently '
              f'({single_axis_order}) than the throughput+RTT+loss composite used here '
              f'({composite_order}) -- at least one cluster trades speed for materially worse loss.')
    return report

def membership_overlap(gmm, X_scaled, working, report, dataset_name):
    '''How cleanly separated are the profiles? For every row, GMM gives a probability of belonging
    to each component -- unlike k-means' hard 0/1 assignment. A row with max probability near 1.0
    clearly belongs to one profile; a row near 1/k is genuinely ambiguous, sitting in the overlap
    between two (or more) profiles.'''
    probs = gmm.predict_proba(X_scaled)
    max_prob = probs.max(axis=1)
    overlap = working.copy()
    overlap['membership_confidence'] = max_prob
    conf_by_cluster = overlap.groupby('cluster')['membership_confidence'].agg(['mean', 'median']).rename(
        columns={'mean': 'avg_confidence', 'median': 'median_confidence'})
    conf_by_cluster = conf_by_cluster.reset_index().merge(report[['cluster', 'profile']], on='cluster').set_index('profile')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(max_prob, bins=40, color=GREY, alpha=.85)
    ax.axvline(max_prob.mean(), color=RED, ls='--', lw=1.5, label=f'Mean = {max_prob.mean():.3f}')
    ax.set(title=f'{dataset_name.title()}: membership confidence (max posterior probability per row)',
           xlabel='Max P(cluster | row)', ylabel='Rows')
    ax.legend(); ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f'{dataset_name}_membership_confidence.png', dpi=180); plt.show()

    ambiguous_share = (max_prob < 0.7).mean()
    print(f'Average max-membership probability across all rows: {max_prob.mean():.3f}')
    print(f'{100*ambiguous_share:.1f}% of rows have max cluster probability < 0.70 -- these sit in the '
          f'overlap between two or more profiles rather than clearly belonging to one.')
    display(conf_by_cluster.round(3))
    return overlap

def ranked_cluster_colours(report):
    '''Maps cluster id -> (profile name, colour), using the shared contrasting CLUSTER_PALETTE
    in ranked (best-to-worst) order -- the same colours used for the profile bar charts above,
    so a given profile always has the same colour everywhere in this notebook.'''
    colours = CLUSTER_PALETTE[:len(report)]
    return {int(row.cluster): (row.profile, colours[i])
            for i, row in report.reset_index(drop=True).iterrows()}

def selected_silhouette(k_eval, chosen_k):
    return float(k_eval.loc[k_eval['k'].eq(chosen_k), 'silhouette'].iloc[0])

def scatter_one_pair(ax, frame, labels, report, x, y, title, xscale='linear', yscale='linear',
                      centroid_space='original', gmm=None):
    '''One 2-D scatter, coloured by GMM cluster assignment (hard label from `predict`), with a
    star marking each component's mean. `centroid_space='model'` plots the star from `gmm.means_`
    directly (for a panel already drawn in scaled log1p space); otherwise it uses the
    `centroid_<x>`/`centroid_<y>` columns in `report` (original units, see build_gmm_cluster_report).'''
    mapping = ranked_cluster_colours(report)
    for cluster, (profile, colour) in mapping.items():
        mask = np.asarray(labels) == cluster
        ax.scatter(frame.loc[mask, x], frame.loc[mask, y], s=8, alpha=.24, color=colour,
                   edgecolors='none', rasterized=True,
                   label=f'{profile} (n={mask.sum():,})')
        row = report.loc[report['cluster'].eq(cluster)].iloc[0]
        if centroid_space == 'original':
            cx, cy = row[f'centroid_{x}'], row[f'centroid_{y}']
        else:
            cx, cy = gmm.means_[cluster, 0], gmm.means_[cluster, 1]
        ax.scatter(cx, cy, marker='*', s=230, color=colour, edgecolor='black', linewidth=.9, zorder=5)
    ax.set(xlabel=x, ylabel=y, title=title, xscale=xscale, yscale=yscale)
    ax.grid(alpha=.18, which='both')
    ax.legend(fontsize=8, markerscale=2, loc='best')

def kmeans_silhouette_at_k(X_scaled, k, sample_size=5000):
    '''Fits k-means on the exact same (log1p + RobustScaler) feature matrix GMM used, at the
    same k, so the two algorithms are compared on identical inputs -- only the clustering method
    differs. Silhouette uses the same fixed random_state/sample_size as the rest of this notebook
    and clustering.ipynb, so the two notebooks' scores are on a comparable basis.'''
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels, sample_size=min(sample_size, len(X_scaled)),
                            random_state=RANDOM_STATE)
    return sil, km


## Download

Features: `download_mbps`, `min_rtt_ms`, `loss_rate`.

In [ ]:
download_gmm_df = prepare_features(download, DOWNLOAD_FEATURES)
download_X, download_scaler = transform_and_scale(download_gmm_df, DOWNLOAD_FEATURES)
download_k_eval, download_models = evaluate_gmm_k_range(download_X)
display(download_k_eval.round(2))
plot_k_selection(download_k_eval, 'download')
download_k = choose_k(download_k_eval)

In [ ]:
download_gmm = download_models[download_k]
download_report, download_working = build_gmm_cluster_report(download_gmm_df, DOWNLOAD_FEATURES, download_gmm, download_X, download_scaler)
download_report = rank_profiles(download_report, throughput_col='download_mbps', loss_col='loss_rate')
display(download_report.round(4))

### Download: three pairwise cluster views

Colours below represent the GMM **hard cluster assignment** (`predict`, mapped to the ranked
profile name via the shared `CLUSTER_PALETTE`), so this is directly comparable to the equivalent
k-means chart in `clustering.ipynb`. Stars mark each component's **mean** (the GMM analogue of a
k-means centroid) back-transformed into original units. With three download clustering features
there are exactly three unique two-axis combinations. Loss uses a symmetric-log scale so that
genuine zero-loss measurements remain visible. GMM was fit in `log1p` + `RobustScaler` space, even
when a chart displays original units.

In [ ]:
download_gmm_silhouette = selected_silhouette(download_k_eval, download_k)
fig, axes = plt.subplots(1, 3, figsize=(20, 5.7))
download_pairs = [
    ('download_mbps', 'min_rtt_ms', 'Throughput vs RTT', 'log', 'log'),
    ('download_mbps', 'loss_rate', 'Throughput vs loss', 'log', 'symlog'),
    ('min_rtt_ms', 'loss_rate', 'RTT vs loss', 'log', 'symlog'),
]
for ax, (x, y, title, xs, ys) in zip(axes, download_pairs):
    scatter_one_pair(ax, download_working, download_working['cluster'].values, download_report, x, y, title, xs, ys)
    if y == 'loss_rate':
        ax.set_yscale('symlog', linthresh=1e-4)
        ax.set_ylim(bottom=0)
fig.suptitle(f'Download GMM assignments — k={download_k}, silhouette={download_gmm_silhouette:.3f}', y=.99, fontsize=15)
fig.tight_layout(rect=[0, 0, 1, .93]); fig.savefig(FIGURE_DIR / 'download_cluster_pairwise_scatter.png', dpi=200, bbox_inches='tight'); plt.show()

### Final presentation table — Download

In [ ]:
download_presentation = download_report[['profile', 'mean_download_mbps', 'median_download_mbps',
                                          'mean_min_rtt_ms', 'median_min_rtt_ms',
                                          'mean_loss_rate', 'median_loss_rate', 'count', 'percent']].rename(columns={
    'mean_download_mbps': 'Mean throughput (Mbps)', 'median_download_mbps': 'Median throughput (Mbps)',
    'mean_min_rtt_ms': 'Mean RTT (ms)', 'median_min_rtt_ms': 'Median RTT (ms)',
    'mean_loss_rate': 'Mean loss (fraction)', 'median_loss_rate': 'Median loss (fraction)',
    'count': 'Count', 'percent': 'Percentage', 'profile': 'Profile'})
download_presentation = download_presentation.round(2)
download_presentation.to_csv(DATA_DIR / 'download_gmm_profiles.csv', index=False)
display(download_presentation)

colors = CLUSTER_PALETTE[:len(download_report)]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(download_report['profile'], download_report['percent'], color=colors)
for y, (pct, mbps) in enumerate(zip(download_report['percent'], download_report['median_download_mbps'])):
    ax.text(pct + 0.5, y, f'{pct:.1f}%  (median {mbps:.1f} Mbps)', va='center', fontsize=9)
ax.set(title='Download network-quality profiles (GMM)', xlabel='% of rows')
ax.invert_yaxis(); ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'download_gmm_profiles.png', dpi=180, bbox_inches='tight'); plt.show()

### Profile overlap — Download

In [ ]:
download_overlap = membership_overlap(download_gmm, download_X, download_working, download_report, 'download')

## Upload

Features: `upload_mbps`, `min_rtt_ms` (no loss feature — upload loss was dropped in `outlier.ipynb`
as all-null metadata).

In [ ]:
upload_gmm_df = prepare_features(upload, UPLOAD_FEATURES)
upload_X, upload_scaler = transform_and_scale(upload_gmm_df, UPLOAD_FEATURES)
upload_k_eval, upload_models = evaluate_gmm_k_range(upload_X)
display(upload_k_eval.round(2))
plot_k_selection(upload_k_eval, 'upload')
upload_k = choose_k(upload_k_eval)

In [ ]:
upload_gmm = upload_models[upload_k]
upload_report, upload_working = build_gmm_cluster_report(upload_gmm_df, UPLOAD_FEATURES, upload_gmm, upload_X, upload_scaler)
upload_report = rank_profiles(upload_report, throughput_col='upload_mbps')
display(upload_report.round(4))

### Upload: three complementary two-axis cluster views

Upload has only two clustering features, so only one unique feature pair exists. To provide three
useful views without inventing a third feature: (1) original units on log axes show the full
tails, (2) a central-99% linear zoom shows overlap near the dense region, and (3) the standardized
`log1p` + `RobustScaler` model space shows the geometry GMM actually fit on. All three show the
same hard-label assignments, coloured with the same contrasting `CLUSTER_PALETTE` as the download
plots and the profile bar charts above.

In [ ]:
upload_gmm_silhouette = selected_silhouette(upload_k_eval, upload_k)
fig, axes = plt.subplots(1, 3, figsize=(20, 5.7))
scatter_one_pair(axes[0], upload_working, upload_working['cluster'].values, upload_report,
                 'upload_mbps', 'min_rtt_ms', 'Full range: original units', 'log', 'log')

scatter_one_pair(axes[1], upload_working, upload_working['cluster'].values, upload_report,
                 'upload_mbps', 'min_rtt_ms', 'Central 99%: original units')
axes[1].set_xlim(0, upload_working['upload_mbps'].quantile(.995))
axes[1].set_ylim(0, upload_working['min_rtt_ms'].quantile(.995))

upload_model_space = pd.DataFrame(upload_X, columns=['robust_scaled_log1p_upload', 'robust_scaled_log1p_rtt'])
scatter_one_pair(axes[2], upload_model_space, upload_working['cluster'].values, upload_report,
                 'robust_scaled_log1p_upload', 'robust_scaled_log1p_rtt',
                 'Model space used by GMM', centroid_space='model', gmm=upload_gmm)
fig.suptitle(f'Upload GMM assignments — k={upload_k}, silhouette={upload_gmm_silhouette:.3f}', y=.99, fontsize=15)
fig.tight_layout(rect=[0, 0, 1, .93]); fig.savefig(FIGURE_DIR / 'upload_cluster_three_views.png', dpi=200, bbox_inches='tight'); plt.show()

### Final presentation table — Upload

In [ ]:
upload_presentation = upload_report[['profile', 'mean_upload_mbps', 'median_upload_mbps',
                                      'mean_min_rtt_ms', 'median_min_rtt_ms', 'count', 'percent']].rename(columns={
    'mean_upload_mbps': 'Mean throughput (Mbps)', 'median_upload_mbps': 'Median throughput (Mbps)',
    'mean_min_rtt_ms': 'Mean RTT (ms)', 'median_min_rtt_ms': 'Median RTT (ms)',
    'count': 'Count', 'percent': 'Percentage', 'profile': 'Profile'})
upload_presentation = upload_presentation.round(2)
upload_presentation.to_csv(DATA_DIR / 'upload_gmm_profiles.csv', index=False)
display(upload_presentation)

colors = CLUSTER_PALETTE[:len(upload_report)]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(upload_report['profile'], upload_report['percent'], color=colors)
for y, (pct, mbps) in enumerate(zip(upload_report['percent'], upload_report['median_upload_mbps'])):
    ax.text(pct + 0.5, y, f'{pct:.1f}%  (median {mbps:.1f} Mbps)', va='center', fontsize=9)
ax.set(title='Upload network-quality profiles (GMM)', xlabel='% of rows')
ax.invert_yaxis(); ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'upload_gmm_profiles.png', dpi=180, bbox_inches='tight'); plt.show()

### Profile overlap — Upload

In [ ]:
upload_overlap = membership_overlap(upload_gmm, upload_X, upload_working, upload_report, 'upload')

## K-means vs. GMM — head-to-head

Same preprocessing (`log1p` + `RobustScaler`), same feature sets, same chosen k, for both download
and upload -- only the clustering algorithm differs. Silhouette score (-1 to 1, higher = tighter
and better-separated clusters) compares the two directly. GMM's soft/probabilistic assignment and
full covariance mixtures are more flexible than k-means' spherical, hard-assignment clusters -- but
that flexibility only pays off if the real cluster shapes actually need it. This chart lets the
data answer that instead of assuming it.

In [ ]:
download_kmeans_sil, download_kmeans_model = kmeans_silhouette_at_k(download_X, download_k)
upload_kmeans_sil, upload_kmeans_model = kmeans_silhouette_at_k(upload_X, upload_k)
download_gmm_sil = selected_silhouette(download_k_eval, download_k)
upload_gmm_sil = selected_silhouette(upload_k_eval, upload_k)

comparison = pd.DataFrame({
    'dataset': ['Download', 'Upload'],
    'k': [download_k, upload_k],
    'kmeans_silhouette': [download_kmeans_sil, upload_kmeans_sil],
    'gmm_silhouette': [download_gmm_sil, upload_gmm_sil],
})
comparison['winner'] = np.where(comparison['kmeans_silhouette'] > comparison['gmm_silhouette'], 'K-means', 'GMM')
display(comparison.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparison))
width = 0.35
bars1 = ax.bar(x - width / 2, comparison['kmeans_silhouette'], width, label='K-means', color=BLUE)
bars2 = ax.bar(x + width / 2, comparison['gmm_silhouette'], width, label='GMM', color=RED)
ax.set_xticks(x); ax.set_xticklabels(comparison['dataset'])
ax.set(ylabel='Silhouette score (same features & k, higher = better separated)',
       title='K-means vs. GMM: silhouette score head-to-head')
ax.legend(); ax.grid(axis='y', alpha=.25)
for bars in (bars1, bars2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + .005, f'{b.get_height():.3f}',
                ha='center', fontsize=9)
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'kmeans_vs_gmm_silhouette.png', dpi=180, bbox_inches='tight'); plt.show()

for _, row in comparison.iterrows():
    print(f"{row['dataset']} (k={row['k']}): K-means silhouette={row['kmeans_silhouette']:.3f}, "
          f"GMM silhouette={row['gmm_silhouette']:.3f} -> {row['winner']} produces better-separated "
          f"clusters here; GMM still adds soft membership/overlap info K-means can't provide.")